# 04 OTU clustering

- **Input:** primer-trimmed reads from `01_preprocessing.ipynb` (`reads_trimmed.qza`)

- **Steps:** 

Quality filtering → Dereplication → Chimera removal → De novo clustering (97%) → Closed-reference clustering against UNITE (97%)

- **Output:** 
    - OTU tables 
    - representative sequences
    - to be compared with the DADA2 ASVs
    
---
extended version

# 04 OTU Clustering

**Creator:** Gwendolin

**Input:**<br>

`processed_ignored/01_preprocessing/`
- primer-trimmed reads (`reads_trimmed.qza`)
<br>

`raw/`
- UNITE ITS reference sequences (`ITS_database.fasta`)
- sample metadata for the OTU table summaries (`fungut_metadata.tsv`)
<br>

`processed_ignored/02_denoise/`
- DADA2 ASV feature table for comparison (`dada2_table.qza`)

**Steps:**<br>

Quality filtering → Dereplication → Chimera removal → two clustering runs from the same non-chimeric sequences and table

- De novo clustering at 97% identity
- Closed-reference clustering against UNITE at 97% identity

**Output:**<br>

`processed_ignored/04_otu/`
- de novo OTU feature table (`otu_table_de_novo_97.qza`)
- de novo OTU representative sequences (`otu_seqs_de_novo_97.qza`)
- closed-reference OTU feature table (`otu_table_closed_ref_97.qza`)
- closed-reference OTU representative sequences (`otu_seqs_closed_ref_97.qza`)
<br>

`processed/04_otu/`
- quality-filtering statistics (`sq_qc_stats.qzv`)
- quality-filtered read summary (`sq_qc_summary.qzv`)
- de novo OTU table summary (`otu_table_de_novo_97.qzv`)
- closed-reference OTU table summary (`otu_table_closed_ref_97.qzv`)


## 0 Setup

In [1]:
import pandas as pd
from qiime2 import Visualization, Artifact

raw_dir  = '../data/raw'
ref_dir  = '../data/raw'
in_dir   = '../data/processed_ignored/01_preprocessing'
out_dir  = '../data/processed_ignored/04_otu'
vis_dir  = '../data/processed/04_otu'

In [2]:
!mkdir -p $out_dir $vis_dir

## 01 Quality filtering
Check in the stats below how many reads per sample are kept.

In [3]:
!qiime quality-filter q-score \
  --i-demux $in_dir/reads_trimmed.qza \
  --p-min-quality 30 \
  --p-min-length-fraction 0.9 \
  --o-filtered-sequences $out_dir/sq_qc.qza \
  --o-filter-stats $out_dir/sq_qc_stats.qza

Saved SampleData[SequencesWithQuality] to: ../data/processed_ignored/04_otu/sq_qc.qza
Saved QualityFilterStats to: ../data/processed_ignored/04_otu/sq_qc_stats.qza


In [4]:
!qiime metadata tabulate \
  --m-input-file $out_dir/sq_qc_stats.qza \
  --o-visualization $vis_dir/sq_qc_stats.qzv

!qiime demux summarize \
  --i-data $out_dir/sq_qc.qza \
  --o-visualization $vis_dir/sq_qc_summary.qzv

Saved Visualization to: ../data/processed/04_otu/sq_qc_stats.qzv
Saved Visualization to: ../data/processed/04_otu/sq_qc_summary.qzv


In [ ]:
Visualization.load(f'{vis_dir}/sq_qc_stats.qzv')

## 02 Dereplication 

In [6]:
!qiime vsearch dereplicate-sequences \
  --i-sequences $out_dir/sq_qc.qza \
  --o-dereplicated-table $out_dir/derep_table.qza \
  --o-dereplicated-sequences $out_dir/derep_seqs.qza

Saved FeatureTable[Frequency] to: ../data/processed_ignored/04_otu/derep_table.qza
Saved FeatureData[Sequence] to: ../data/processed_ignored/04_otu/derep_seqs.qza


## 03 Chimera removal

In [7]:
!qiime vsearch uchime-denovo \
  --i-sequences $out_dir/derep_seqs.qza \
  --i-table $out_dir/derep_table.qza \
  --o-chimeras $out_dir/chimeras.qza \
  --o-nonchimeras $out_dir/nonchimeras.qza \
  --o-stats $out_dir/chimera_stats.qza

Saved FeatureData[Sequence] to: ../data/processed_ignored/04_otu/chimeras.qza
Saved FeatureData[Sequence] to: ../data/processed_ignored/04_otu/nonchimeras.qza
Saved UchimeStats to: ../data/processed_ignored/04_otu/chimera_stats.qza


In [8]:
# keep only non-chimeric sequences in the table and in the sequences
!qiime feature-table filter-features \
  --i-table $out_dir/derep_table.qza \
  --m-metadata-file $out_dir/nonchimeras.qza \
  --o-filtered-table $out_dir/derep_table_nochim.qza

!qiime feature-table filter-seqs \
  --i-data $out_dir/derep_seqs.qza \
  --m-metadata-file $out_dir/nonchimeras.qza \
  --o-filtered-data $out_dir/derep_seqs_nochim.qza

Saved FeatureTable[Frequency] to: ../data/processed_ignored/04_otu/derep_table_nochim.qza
Saved FeatureData[Sequence] to: ../data/processed_ignored/04_otu/derep_seqs_nochim.qza


## 04  De novo clustering (97 %)

In [9]:
!qiime vsearch cluster-features-de-novo \
  --i-table $out_dir/derep_table_nochim.qza \
  --i-sequences $out_dir/derep_seqs_nochim.qza \
  --p-perc-identity $0.97 \
  --p-threads 3 \
  --o-clustered-table $out_dir/otu_table_de_novo_97.qza \
  --o-clustered-sequences $out_dir/otu_seqs_de_novo_97.qza

Saved FeatureTable[Frequency] to: ../data/processed_ignored/04_otu/otu_table_de_novo_97.qza
Saved FeatureData[Sequence] to: ../data/processed_ignored/04_otu/otu_seqs_de_novo_97.qza


## 05 Closed-reference clustering against UNITE (97 %)

In [10]:
ref_fasta = f'{ref_dir}/ITS_database.fasta'

In [11]:
!qiime tools import \
  --type 'FeatureData[Sequence]' \
  --input-path $ref_fasta \
  --output-path $ref_dir/unite_97_refs.qza

Imported ../data/raw/ITS_database.fasta as DNASequencesDirectoryFormat to ../data/raw/unite_97_refs.qza


In [2]:
!qiime vsearch cluster-features-closed-reference \
  --i-table $out_dir/derep_table_nochim.qza \
  --i-sequences $out_dir/derep_seqs_nochim.qza \
  --i-reference-sequences $ref_dir/unite_97_refs.qza \
  --p-perc-identity $0.97 \
  --p-threads 3 \
  --o-clustered-table $out_dir/otu_table_closed_ref_97.qza \
  --o-clustered-sequences $out_dir/otu_seqs_closed_ref_97.qza \
  --o-unmatched-sequences $out_dir/otu_unmatched_closed_ref_97.qza

Saved FeatureTable[Frequency] to: ../data/processed_ignored/04_otu/otu_table_closed_ref_97.qza
Saved FeatureData[Sequence] to: ../data/processed_ignored/04_otu/otu_seqs_closed_ref_97.qza
Saved FeatureData[Sequence] to: ../data/processed_ignored/04_otu/otu_unmatched_closed_ref_97.qza


## 06 Summary

In [3]:
!qiime feature-table summarize \
  --i-table $out_dir/otu_table_de_novo_97.qza \
  --m-metadata-file $raw_dir/fungut_metadata.tsv \
  --o-feature-frequencies $out_dir/otu_table_de_novo_97_feature_frequencies.qza \
  --o-sample-frequencies $out_dir/otu_table_de_novo_97_sample_frequencies.qza \
  --o-summary $vis_dir/otu_table_de_novo_97.qzv

!qiime feature-table tabulate-seqs \
  --i-data $out_dir/otu_seqs_de_novo_97.qza \
  --o-visualization $vis_dir/otu_seqs_de_novo_97.qzv

Saved ImmutableMetadata to: ../data/processed_ignored/04_otu/otu_table_de_novo_97_feature_frequencies.qza
Saved ImmutableMetadata to: ../data/processed_ignored/04_otu/otu_table_de_novo_97_sample_frequencies.qza
Saved Visualization to: ../data/processed/04_otu/otu_table_de_novo_97.qzv
Saved Visualization to: ../data/processed/04_otu/otu_seqs_de_novo_97.qzv


In [ ]:
Visualization.load(f'{vis_dir}/otu_table_de_novo_97.qzv')

In [5]:
!qiime feature-table summarize \
  --i-table $out_dir/otu_table_closed_ref_97.qza \
  --m-metadata-file $raw_dir/fungut_metadata.tsv \
  --o-feature-frequencies $out_dir/otu_table_closed_ref_97_feature_frequencies.qza \
  --o-sample-frequencies $out_dir/otu_table_closed_ref_97_sample_frequencies.qza \
  --o-summary $vis_dir/otu_table_closed_ref_97.qzv

!qiime feature-table tabulate-seqs \
  --i-data $out_dir/otu_seqs_closed_ref_97.qza \
  --o-visualization $vis_dir/otu_seqs_closed_ref_97.qzv

Saved ImmutableMetadata to: ../data/processed_ignored/04_otu/otu_table_closed_ref_97_feature_frequencies.qza
Saved ImmutableMetadata to: ../data/processed_ignored/04_otu/otu_table_closed_ref_97_sample_frequencies.qza
Saved Visualization to: ../data/processed/04_otu/otu_table_closed_ref_97.qzv
Saved Visualization to: ../data/processed/04_otu/otu_seqs_closed_ref_97.qzv


In [ ]:
Visualization.load(f'{vis_dir}/otu_table_closed_ref_97.qzv')

## 08 Comparison with DADA2

In [6]:
tables = {
    'ASV (DADA2)':        '../data/processed_ignored/02_denoise/dada2_table.qza',
    'OTU de novo 97%':    f'{out_dir}/otu_table_de_novo_97.qza',
    'OTU closed-ref 97%': f'{out_dir}/otu_table_closed_ref_97.qza',
}

rows = []
for name, path in tables.items():
    df = Artifact.load(path).view(pd.DataFrame)   # rows = samples, columns = features
    rows.append({
        'method': name,
        'features': df.shape[1],
        'total reads': int(df.values.sum()),
        'median reads per sample': int(df.sum(axis=1).median()),
    })

pd.DataFrame(rows)

,method,features,total reads,median reads per sample
0,ASV (DADA2),681,9037812,62304
1,OTU de novo 97%,25949,6959156,46550
2,OTU closed-ref 97%,4395,6791249,45559
